# Word Embedding Clustering with KMeans

This notebook demonstrates that **clustering pre-trained word embeddings** can group
**semantically related words** into the same cluster.

We will:

1. Load pre-trained **GloVe** word embeddings using `gensim`.
2. Choose a small set of words from a few semantic groups (aircraft, animals, foods, tools).
3. Run **KMeans** on the word embeddings.
4. Visualize the clusters in 2D using **PCA**.

You can extend this notebook by:
- Adding more words
- Trying different numbers of clusters
- Using other dimensionality reduction methods like t-SNE or UMAP


In [ ]:
# Imports
import numpy as np
import gensim.downloader as api
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Make plots a bit nicer
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = True

In [ ]:
# Load pre-trained GloVe embeddings
# This may take a minute the first time as it downloads the model.
glove_vectors = api.load("glove-wiki-gigaword-100")

print(f"Loaded GloVe with {len(glove_vectors.key_to_index)} words "
      f"and {glove_vectors.vector_size}-dimensional embeddings.")

In [ ]:
# Choose words from a few semantic groups
candidate_words = [
    # aircraft
    "airplane", "plane", "jet", "airport", "runway", "helicopter",
    # animals
    "dog", "cat", "wolf", "tiger", "lion", "cheetah",
    # foods
    "pizza", "burger", "sandwich", "pasta", "salad", "sushi",
    # tools
    "hammer", "screwdriver", "wrench", "saw", "drill"
]

# Keep only words that exist in the embedding vocabulary
words = [w for w in candidate_words if w in glove_vectors.key_to_index]

print("Words used (present in GloVe vocabulary):")
for w in words:
    print(" -", w)

In [ ]:
# Build an embedding matrix X where each row is a word vector
X = np.stack([glove_vectors[w] for w in words])
print("Embedding matrix shape:", X.shape)

In [ ]:
# Cluster the word embeddings with KMeans
num_clusters = 4  # we expect roughly 4 groups: aircraft, animals, food, tools
kmeans = KMeans(n_clusters=num_clusters, random_state=0, n_init="auto")
labels = kmeans.fit_predict(X)

print("Cluster assignments:")
for word, label in zip(words, labels):
    print(f"{word:12s} -> cluster {label}")

In [ ]:
# Visualize the clusters in 2D using PCA
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

plt.figure()
for cluster_id in range(num_clusters):
    mask = (labels == cluster_id)
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], label=f"Cluster {cluster_id}")
    
    # Label each point with its word
    for x, y, w in zip(X_2d[mask, 0], X_2d[mask, 1], np.array(words)[mask]):
        plt.text(x + 0.02, y + 0.02, w, fontsize=9)

plt.title("KMeans clustering of GloVe word embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()